## Split Abstracts by Category

Splits `abstracts.jsonl` (all ~50k-56k ArXiv CS abstracts) into 10 category-wise JSONL files, one per subcategory, using `primary_cat` as the bucketing key.

**Why `primary_cat` and not `categories`**: a paper's `categories` field can list several tags (e.g. `"cs.AI cs.LG cs.NE"`), but each paper needs exactly one home for a clean category-wise split with no duplication. `primary_cat` is the single tag ArXiv itself considers the paper's main category, so it's the right key for a partition (as opposed to a multi-label filter).

**Categories** (10 total, split between you and your collaborator):
- Yours: `cs.AI`, `cs.LG`, `cs.IR`, `cs.DB`, `cs.SE`
- Collaborator's: `cs.CV`, `cs.CL`, `cs.NE`, `cs.DC`, `cs.CR`

Papers whose `primary_cat` falls outside these 10 are written to a separate `_other.jsonl` file so nothing silently disappears; you can inspect and decide whether to fold them in or drop them.

### Configuration

In [1]:
import json
import os
from collections import defaultdict

# Path to the combined abstracts file (~78MB, all papers together)
INPUT_PATH = "../data/arxiv/abstracts.jsonl"

# Output directory for the 10 category-wise files
OUTPUT_DIR = "../data/arxiv/abstracts/by_category/"

# The 10 target categories. Anything outside this set goes to _other.jsonl
TARGET_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

### Load and validate

Streams the file line-by-line rather than `json.load()`-ing the whole thing at once, since 78MB of JSONL as one giant list is wasteful when we only need to bucket and re-write it. Malformed lines are logged and skipped rather than crashing the whole run.

In [2]:
buckets = defaultdict(list)
malformed_lines = 0
missing_primary_cat = 0
total_lines = 0

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        total_lines += 1

        try:
            record = json.loads(line)
        except json.JSONDecodeError as e:
            malformed_lines += 1
            print(f"[WARN] line {line_num}: could not parse JSON ({e}), skipping")
            continue

        primary_cat = record.get("primary_cat")
        if not primary_cat:
            missing_primary_cat += 1
            print(f"[WARN] line {line_num}: missing 'primary_cat' (id={record.get('id', '?')}), routing to _other")
            buckets["_other"].append(record)
            continue

        if primary_cat in TARGET_CATEGORIES:
            buckets[primary_cat].append(record)
        else:
            buckets["_other"].append(record)

print(f"\nTotal lines read: {total_lines}")
print(f"Malformed lines skipped: {malformed_lines}")
print(f"Missing primary_cat: {missing_primary_cat}")


Total lines read: 25004
Malformed lines skipped: 0
Missing primary_cat: 0


### Check for duplicate IDs within each bucket

A duplicate `id` inside the same category file would silently inflate your later sampling (e.g. W1 generation could draw the same abstract twice). This does not modify data, it only reports so you can decide how to handle it if any turn up.

In [3]:
for category, records in buckets.items():
    ids = [r.get("id") for r in records]
    seen = set()
    dupes = set()
    for _id in ids:
        if _id in seen:
            dupes.add(_id)
        seen.add(_id)
    if dupes:
        print(f"[WARN] {category}: {len(dupes)} duplicate id(s) found, e.g. {list(dupes)[:5]}")
    else:
        print(f"{category}: no duplicate ids ({len(records)} records)")

cs.AI: no duplicate ids (7369 records)
cs.LG: no duplicate ids (10746 records)
cs.DB: no duplicate ids (1532 records)
cs.SE: no duplicate ids (2281 records)
cs.IR: no duplicate ids (3076 records)


### Write category-wise files

One JSONL file per category, e.g. `cs.AI.jsonl`, `cs.IR.jsonl`, etc. `_other.jsonl` is written too if any records landed there so nothing is lost silently.

In [4]:
summary = {}

for category, records in buckets.items():
    out_path = os.path.join(OUTPUT_DIR, f"{category}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    summary[category] = len(records)
    print(f"Wrote {len(records):>6} records -> {out_path}")

Wrote   7369 records -> ../data/arxiv/abstracts/by_category/cs.AI.jsonl
Wrote  10746 records -> ../data/arxiv/abstracts/by_category/cs.LG.jsonl
Wrote   1532 records -> ../data/arxiv/abstracts/by_category/cs.DB.jsonl
Wrote   2281 records -> ../data/arxiv/abstracts/by_category/cs.SE.jsonl
Wrote   3076 records -> ../data/arxiv/abstracts/by_category/cs.IR.jsonl


### Verify the split

Confirms every record from the input landed in exactly one output file (counts sum back to `total_lines` minus malformed lines), and shows the per-category distribution so you can sanity-check nothing is empty or wildly skewed before moving to the next stage.

In [5]:
total_written = sum(summary.values())
expected = total_lines - malformed_lines

print("Per-category counts:")
for category in TARGET_CATEGORIES:
    count = summary.get(category, 0)
    flag = "  <-- EMPTY, check this" if count == 0 else ""
    print(f"  {category:8s}: {count:>6}{flag}")

if "_other" in summary:
    print(f"  {'_other':8s}: {summary['_other']:>6}  (outside the 10 target categories)")

print(f"\nTotal written: {total_written}")
print(f"Expected (valid lines read): {expected}")
print("OK: counts match" if total_written == expected else "MISMATCH: investigate before proceeding")

Per-category counts:
  cs.AI   :   7369
  cs.LG   :  10746
  cs.IR   :   3076
  cs.DB   :   1532
  cs.SE   :   2281
  cs.CV   :      0  <-- EMPTY, check this
  cs.CL   :      0  <-- EMPTY, check this
  cs.NE   :      0  <-- EMPTY, check this
  cs.DC   :      0  <-- EMPTY, check this
  cs.CR   :      0  <-- EMPTY, check this

Total written: 25004
Expected (valid lines read): 25004
OK: counts match
